In [1]:
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

In [2]:
def preprocess_document(raw_text: str, filepath: str) -> Dict[str, Any]:
    """
    Preprocess một tài liệu: extract metadata từ header và làm sạch nội dung.

    Args:
        raw_text: Toàn bộ nội dung file text
        filepath: Đường dẫn file để làm source mặc định

    Returns:
        Dict chứa:
          - "text": nội dung đã clean
          - "metadata": dict với source, department, effective_date, access

    TODO Sprint 1:
    - Extract metadata từ dòng đầu file (Source, Department, Effective Date, Access)
    - Bỏ các dòng header metadata khỏi nội dung chính
    - Normalize khoảng trắng, xóa ký tự rác

    Gợi ý: dùng regex để parse dòng "Key: Value" ở đầu file.
    """
    lines = raw_text.strip().split("\n")
    metadata = {
        "source": filepath,
        "section": "",
        "department": "unknown",
        "effective_date": "unknown",
        "access": "internal",
    }
    content_lines = []
    header_done = False

    for line in lines:
        if not header_done:
            # TODO: Parse metadata từ các dòng "Key: Value"
            # Ví dụ: "Source: policy/refund-v4.pdf" → metadata["source"] = "policy/refund-v4.pdf"
            if line.startswith("Source:"):
                metadata["source"] = line.replace("Source:", "").strip()
            elif line.startswith("Department:"):
                metadata["department"] = line.replace("Department:", "").strip()
            elif line.startswith("Effective Date:"):
                metadata["effective_date"] = line.replace("Effective Date:", "").strip()
            elif line.startswith("Access:"):
                metadata["access"] = line.replace("Access:", "").strip()
            elif line.startswith("==="):
                # Gặp section heading đầu tiên → kết thúc header
                header_done = True
                content_lines.append(line)
            elif line.strip() == "" or line.isupper():
                # Dòng tên tài liệu (toàn chữ hoa) hoặc dòng trống
                continue
        else:
            content_lines.append(line)

    cleaned_text = "\n".join(content_lines)
    # Normalize text: bỏ ký tự đặc biệt thừa, chuẩn hóa dấu câu
    # Giảm số dòng trống liên tiếp xuống tối đa 2 dòng
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
    # Loại bỏ khoảng trắng thừa (multiple spaces → single space)
    cleaned_text = re.sub(r" {2,}", " ", cleaned_text)
    # Chuẩn hóa khoảng trắng xung quanh dấu chấm câu
    cleaned_text = re.sub(r"\s+([.,!?;:])", r"\1", cleaned_text)
    # Xóa khoảng trắng dư thừa ở đầu/cuối mỗi dòng
    cleaned_text = "\n".join(line.rstrip() for line in cleaned_text.split("\n"))
    cleaned_text = cleaned_text.strip()

    return {
        "text": cleaned_text,
        "metadata": metadata,
    }

In [7]:
CHUNK_SIZE = 400      
CHUNK_OVERLAP = 80 
def chunk_document(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Chunk một tài liệu đã preprocess thành danh sách các chunk nhỏ.

    Args:
        doc: Dict với "text" và "metadata" (output của preprocess_document)

    Returns:
        List các Dict, mỗi dict là một chunk với:
          - "text": nội dung chunk
          - "metadata": metadata gốc + "section" của chunk đó

    Quy trình:
    1. Split theo heading "=== Section ... ===" hoặc "=== Phần ... ===" trước
    2. Nếu section quá dài (> CHUNK_SIZE * 4 ký tự), split tiếp theo paragraph
    3. Thêm overlap: lấy đoạn cuối của chunk trước vào đầu chunk tiếp theo
    4. Mỗi chunk giữ metadata đầy đủ từ tài liệu gốc

    Ưu tiên: Cắt tại ranh giới tự nhiên (section, paragraph, câu)
    thay vì cắt theo token count cứng.
    """
    text = doc["text"]
    base_metadata = doc["metadata"].copy()
    chunks = []

    # Bước 1: Split theo heading pattern "=== ... ==="
    sections = re.split(r"(===.*?===)", text)

    current_section = "General"
    current_section_text = ""

    for part in sections:
        if re.match(r"===.*?===", part):
            # Lưu section trước (nếu có nội dung)
            if current_section_text.strip():
                section_chunks = _split_by_size(
                    current_section_text.strip(),
                    base_metadata=base_metadata,
                    section=current_section,
                )
                chunks.extend(section_chunks)
            # Bắt đầu section mới
            current_section = part.strip("= \n")
            current_section_text = ""
        else:
            current_section_text += part

    # Lưu section cuối cùng
    if current_section_text.strip():
        section_chunks = _split_by_size(
            current_section_text.strip(),
            base_metadata=base_metadata,
            section=current_section,
        )
        chunks.extend(section_chunks)

    return chunks


def _split_by_size(
    text: str,
    base_metadata: Dict,
    section: str,
    chunk_chars: int = CHUNK_SIZE * 4,
    overlap_chars: int = CHUNK_OVERLAP * 4,
) -> List[Dict[str, Any]]:
    """
    Helper: Split text dài thành chunks với overlap.

    Cải thiện: split theo paragraph (\n\n) trước, rồi ghép đến khi đủ size.
    Tìm ranh giới tự nhiên (dấu xuống dòng, dấu chấm) khi cắt chunk.
    """
    if len(text) <= chunk_chars:
        # Toàn bộ section vừa một chunk
        return [{
            "text": text,
            "metadata": {**base_metadata, "section": section},
        }]

    # Split theo paragraph (\n\n)
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""
    overlap_buffer = ""  # Dùng để lưu overlap từ chunk trước

    for para in paragraphs:
        # Nếu thêm paragraph này vào sẽ vượt chunk_chars
        if len(current_chunk) + len(para) + 2 > chunk_chars and current_chunk.strip():
            # Lưu chunk hiện tại
            chunks.append({
                "text": (overlap_buffer + current_chunk).strip(),
                "metadata": {**base_metadata, "section": section},
            })
            
            # Chuẩn bị overlap: lấy phần cuối của chunk để làm ngữ cảnh cho chunk tiếp theo
            # Tìm câu cuối cùng trong chunk để cắt một cách tự nhiên
            sentences = re.split(r'(?<=[.!?])\s+', current_chunk)
            if len(sentences) > 1:
                # Lấy 1-2 câu cuối làm overlap
                overlap_buffer = " ".join(sentences[-1:]) + "\n\n"
            else:
                # Nếu không có dấu câu, lấy 30% cuối của chunk
                overlap_size = len(current_chunk) // 3
                overlap_buffer = current_chunk[-overlap_size:].lstrip() + "\n\n"
            
            current_chunk = para
        else:
            # Thêm paragraph vào chunk hiện tại
            if current_chunk:
                current_chunk += "\n\n" + para
            else:
                current_chunk = para

    # Lưu chunk cuối cùng
    if current_chunk.strip():
        chunks.append({
            "text": (overlap_buffer + current_chunk).strip(),
            "metadata": {**base_metadata, "section": section},
        })

    return chunks

In [ ]:

#Đọc file mẫu và chạy thử preprocess_document
input_path = r"C:\Users\letung373\Desktop\2A202600111-LeVanTung-Day08\day08\lab\data\docs\hr_leave_policy.txt"
with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
result = preprocess_document(raw_text, input_path)
print("Metadata:", result["metadata"])
print("--- Nội dung đã clean (preview) ---")
print(result["text"][:500])

Metadata: {'source': 'hr/leave-policy-2026.pdf', 'section': '', 'department': 'HR', 'effective_date': '2026-01-01', 'access': 'internal'}
--- Nội dung đã clean (preview) ---
=== Phần 1: Các loại nghỉ phép ===

1.1 Nghỉ phép năm (Annual Leave):
- Số ngày: 12 ngày/năm cho nhân viên dưới 3 năm kinh nghiệm.
- Số ngày: 15 ngày/năm cho nhân viên từ 3-5 năm kinh nghiệm.
- Số ngày: 18 ngày/năm cho nhân viên trên 5 năm kinh nghiệm.
- Chuyển năm sau: Tối đa 5 ngày phép năm chưa dùng được chuyển sang năm tiếp theo.

1.2 Nghỉ ốm (Sick Leave):
- Số ngày: 10 ngày/năm có trả lương.
- Yêu cầu: Thông báo cho Line Manager trước 9:00 sáng ngày nghỉ.
- Nếu nghỉ trên 3 ngày liên tiếp: C


In [8]:
# Test chunk_document với output của preprocess_document
chunks = chunk_document(result)
print(f"Số lượng chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[Chunk {i+1}] Section: {chunk['metadata']['section']}")
    print(f"Text: {chunk['text'][:200]}...")

Số lượng chunks: 5

[Chunk 1] Section: Phần 1: Các loại nghỉ phép
Text: 1.1 Nghỉ phép năm (Annual Leave):
- Số ngày: 12 ngày/năm cho nhân viên dưới 3 năm kinh nghiệm.
- Số ngày: 15 ngày/năm cho nhân viên từ 3-5 năm kinh nghiệm.
- Số ngày: 18 ngày/năm cho nhân viên trên 5 ...

[Chunk 2] Section: Phần 2: Quy trình xin nghỉ phép
Text: Bước 1: Nhân viên gửi yêu cầu nghỉ phép qua hệ thống HR Portal (https://hr.company.internal) ít nhất 3 ngày làm việc trước ngày nghỉ.
Bước 2: Line Manager phê duyệt hoặc từ chối trong vòng 1 ngày làm ...

[Chunk 3] Section: Phần 3: Chính sách làm thêm giờ
Text: 3.1 Điều kiện làm thêm:
Làm thêm giờ phải được Line Manager phê duyệt trước bằng văn bản.

3.2 Hệ số lương làm thêm:
- Ngày thường: 150% lương giờ tiêu chuẩn.
- Ngày cuối tuần: 200% lương giờ tiêu chu...
